# Notebook 02 — Data Cleaning & ETL Pipeline

**Project:** Why Startups Fail — VC Investment Pattern Analysis  
**Team:** Section C, Group 17  

---
**Objective:** Build a fully documented, reproducible Python ETL pipeline that transforms the raw VC investment dataset into a clean, analysis-ready file. Every transformation is logged with a reason.

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
# Update RAW_PATH to wherever your CSV lives.
# If running on Google Colab, upload the file and use '/content/investments_VC.csv'
RAW_PATH       = '/content/investments_VC (1).csv'          # ← change if needed
CLEAN_PATH     = 'startups_cleaned.csv'            # output file

# ── ETL Audit Log ─────────────────────────────────────────────────────────────
etl_log = []

def log_step(step: int, desc: str, before: int, after: int, detail: str = ""):
    """Append a row to the audit log and print a summary line."""
    dropped = before - after
    pct_kept = after / before * 100 if before > 0 else 0
    etl_log.append({
        "Step"       : step,
        "Description": desc,
        "Before"     : before,
        "After"      : after,
        "Dropped"    : dropped,
        "% Kept"     : round(pct_kept, 1),
        "Detail"     : detail,
    })
    flag = "⚠️" if dropped > 500 else "✅"
    print(f"{flag} [Step {step:02d}] {desc}")
    print(f"          {before:,} → {after:,} rows  |  dropped {dropped:,}  |  {pct_kept:.1f}% retained")

print("Configuration complete.")

Configuration complete.


In [5]:
df = pd.read_csv(RAW_PATH, encoding='latin-1', low_memory=False)
initial_rows = len(df)

print(f"Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names (raw):")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst 3 rows:")
df.head(3)

Raw dataset loaded: 54,294 rows × 39 columns

Column names (raw):
['permalink', 'name', 'homepage_url', 'category_list', ' market ', ' funding_total_usd ', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']

Data types:
permalink                object
name                     object
homepage_url             object
category_list            object
 market                  object
 funding_total_usd       object
status                   object
country_code             object
state_code               object
region                   object
city                     object

,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# ── Initial Missing Value Audit ────────────────────────────────────────────────
print("=== INITIAL MISSING VALUE AUDIT ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
audit = pd.DataFrame({'Missing Count': missing, '% Missing': missing_pct})
print(audit[audit['Missing Count'] > 0].to_string())

=== INITIAL MISSING VALUE AUDIT ===
                      Missing Count  % Missing
permalink                      4856        8.9
name                           4857        8.9
homepage_url                   8305       15.3
category_list                  8817       16.2
 market                        8824       16.3
 funding_total_usd             4856        8.9
status                         6170       11.4
country_code                  10129       18.7
state_code                    24133       44.4
region                        10129       18.7
city                          10972       20.2
funding_rounds                 4856        8.9
founded_at                    15740       29.0
founded_month                 15812       29.1
founded_quarter               15812       29.1
founded_year                  15812       29.1
first_funding_at               4856        8.9
last_funding_at                4856        8.9
seed                           4856        8.9
venture                 

In [7]:
before = len(df)

original_cols = df.columns.tolist()
df.columns = df.columns.str.strip()
renamed = [(o, n) for o, n in zip(original_cols, df.columns.tolist()) if o != n]

log_step(1, 'Strip whitespace from column names', before, len(df),
         f'Renamed columns: {renamed}')

print(f"\nColumns renamed: {renamed}")

✅ [Step 01] Strip whitespace from column names
          54,294 → 54,294 rows  |  dropped 0  |  100.0% retained

Columns renamed: [(' market ', 'market'), (' funding_total_usd ', 'funding_total_usd')]


In [8]:
before = len(df)

def clean_funding(val):
    """Convert a funding string like '1,750,000' or ' - ' to float. Returns NaN for unknowns."""
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(',', '').replace(' ', '')
    if val_str in ['-', '', 'nan', 'None', 'N/A']:
        return np.nan
    try:
        return float(val_str)
    except ValueError:
        return np.nan

df['funding_total_usd'] = df['funding_total_usd'].apply(clean_funding)

log_step(2, 'Convert funding_total_usd to numeric', before, len(df),
         'Stripped commas, dashes, spaces; converted to float64; dashes → NaN')

print(f"\nFunding stats after conversion:")
print(df['funding_total_usd'].describe().apply(lambda x: f'{x:,.0f}'))

✅ [Step 02] Convert funding_total_usd to numeric
          54,294 → 54,294 rows  |  dropped 0  |  100.0% retained

Funding stats after conversion:
count            40,907
mean         15,912,526
std         168,678,800
min                   1
25%             350,000
50%           2,000,000
75%          10,000,000
max      30,079,503,000
Name: funding_total_usd, dtype: object


In [9]:
before = len(df)

date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

log_step(3, 'Parse date columns to datetime64', before, len(df),
         f'Columns converted: {date_cols}. Unparseable strings coerced to NaT.')

print("\nDate column null counts after parsing:")
print(df[date_cols].isnull().sum())

✅ [Step 03] Parse date columns to datetime64
          54,294 → 54,294 rows  |  dropped 0  |  100.0% retained

Date column null counts after parsing:
founded_at          15741
first_funding_at     4866
last_funding_at      4862
dtype: int64


In [10]:
before = len(df)

# Rule A: first funding must be on or after founding date
mask_a = (
    df['first_funding_at'].isna() |
    df['founded_at'].isna() |
    (df['first_funding_at'] >= df['founded_at'])
)
df = df[mask_a]

# Rule B: last funding must be on or after first funding
mask_b = (
    df['last_funding_at'].isna() |
    df['first_funding_at'].isna() |
    (df['last_funding_at'] >= df['first_funding_at'])
)
df = df[mask_b]

log_step(4, 'Enforce date logical ordering', before, len(df),
         'Removed rows where first_funding < founded_at, or last_funding < first_funding')

⚠️ [Step 04] Enforce date logical ordering
          54,294 → 51,555 rows  |  dropped 2,739  |  95.0% retained


In [11]:
before = len(df)

str_cols = ['name', 'market', 'status', 'country_code', 'state_code',
            'region', 'city', 'category_list', 'permalink', 'homepage_url']

null_markers = {'nan', 'none', 'n/a', 'na', '', 'null', '-'}

for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].apply(lambda x: np.nan if x.lower() in null_markers else x)

log_step(5, 'Standardise string columns and unify null markers', before, len(df),
         f'Cleaned {len(str_cols)} columns. Null markers unified to NaN.')

print("\nPost-step sample — unique status values:")
print(df['status'].value_counts(dropna=False))

✅ [Step 05] Standardise string columns and unify null markers
          51,555 → 51,555 rows  |  dropped 0  |  100.0% retained

Post-step sample — unique status values:
status
operating    39584
NaN           6122
acquired      3510
closed        2339
Name: count, dtype: int64


In [12]:
before = len(df)

print("Raw status distribution (before standardisation):")
print(df['status'].value_counts(dropna=False).to_string())

df['status'] = df['status'].str.lower().str.strip()

status_map = {
    'operating': 'operating',
    'closed'   : 'closed',
    'acquired' : 'acquired',
    'ipo'      : 'ipo',
}
df['status'] = df['status'].map(status_map)  # unmapped values → NaN

log_step(6, 'Map status to 4 canonical categories', before, len(df),
         'Values not in {operating, closed, acquired, ipo} set to NaN for next step')

print("\nCleaned status distribution:")
print(df['status'].value_counts(dropna=False).to_string())

Raw status distribution (before standardisation):
status
operating    39584
NaN           6122
acquired      3510
closed        2339
✅ [Step 06] Map status to 4 canonical categories
          51,555 → 51,555 rows  |  dropped 0  |  100.0% retained

Cleaned status distribution:
status
operating    39584
NaN           6122
acquired      3510
closed        2339


In [13]:
before = len(df)

df = df.dropna(subset=['status'])

log_step(7, 'Drop rows with missing status (target variable)', before, len(df),
         'Cannot impute or infer the outcome label for the "Why Startups Fail" analysis')

⚠️ [Step 07] Drop rows with missing status (target variable)
          51,555 → 45,433 rows  |  dropped 6,122  |  88.1% retained


In [14]:
before = len(df)

dedup_key = ['name', 'country_code', 'founded_year']
df = df.drop_duplicates(subset=dedup_key, keep='first')

log_step(8, f'Remove duplicates on {dedup_key}', before, len(df),
         'Kept first occurrence. Duplicates indicate same startup entered multiple times.')

✅ [Step 08] Remove duplicates on ['name', 'country_code', 'founded_year']
          45,433 → 45,420 rows  |  dropped 13  |  100.0% retained


In [15]:
before = len(df)

round_cols = [
    'seed', 'venture', 'equity_crowdfunding', 'undisclosed',
    'convertible_note', 'debt_financing', 'angel', 'grant',
    'private_equity', 'post_ipo_equity', 'post_ipo_debt',
    'secondary_market', 'product_crowdfunding',
    'round_A', 'round_B', 'round_C', 'round_D',
    'round_E', 'round_F', 'round_G', 'round_H'
]
round_cols_present = [c for c in round_cols if c in df.columns]

for col in round_cols_present:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

log_step(9, 'Convert funding round columns to int (NaN → 0)', before, len(df),
         f'{len(round_cols_present)} round columns standardised. NaN = no funding received = 0.')

print(f"\nRound columns processed: {round_cols_present}")

✅ [Step 09] Convert funding round columns to int (NaN → 0)
          45,420 → 45,420 rows  |  dropped 0  |  100.0% retained

Round columns processed: ['seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H']


In [16]:
before = len(df)

Q1  = df['funding_total_usd'].quantile(0.25)
Q3  = df['funding_total_usd'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 3 * IQR

outliers_above = (df['funding_total_usd'] > upper_fence).sum()
df['funding_total_usd'] = df['funding_total_usd'].clip(upper=upper_fence)

log_step(10, 'Cap funding outliers at Q3 + 3×IQR', before, len(df),
         f'Q1={Q1:,.0f}, Q3={Q3:,.0f}, IQR={IQR:,.0f}. '
         f'Upper fence={upper_fence:,.0f}. Capped {outliers_above} rows (not dropped).')

print(f"\nOutlier cap applied: ${upper_fence:,.0f}")
print(f"Values capped: {outliers_above}")
print(f"\nPost-cap funding stats:")
print(df['funding_total_usd'].describe().apply(lambda x: f'{x:,.0f}'))

✅ [Step 10] Cap funding outliers at Q3 + 3×IQR
          45,420 → 45,420 rows  |  dropped 0  |  100.0% retained

Outlier cap applied: $38,800,000
Values capped: 3178

Post-cap funding stats:
count        37,696
mean      8,076,516
std      11,986,974
min               1
25%         400,000
50%       2,000,000
75%      10,000,000
max      38,800,000
Name: funding_total_usd, dtype: object


In [17]:
before = len(df)

YEAR_MIN, YEAR_MAX = 1990, 2014
df = df[
    df['founded_year'].isna() |
    ((df['founded_year'] >= YEAR_MIN) & (df['founded_year'] <= YEAR_MAX))
]

log_step(11, f'Filter to founded year {YEAR_MIN}–{YEAR_MAX}', before, len(df),
         'Pre-1990 data is sparse and structurally different. Max year in dataset is 2014.')

print(f"\nFounded year distribution after filter:")
print(df['founded_year'].value_counts().sort_index().to_string())

⚠️ [Step 11] Filter to founded year 1990–2014
          45,420 → 44,493 rows  |  dropped 927  |  98.0% retained

Founded year distribution after filter:
founded_year
1990.0      83
1991.0      89
1992.0     111
1993.0     121
1994.0     153
1995.0     212
1996.0     293
1997.0     352
1998.0     422
1999.0     734
2000.0     865
2001.0     722
2002.0     768
2003.0     921
2004.0    1119
2005.0    1351
2006.0    1683
2007.0    2152
2008.0    2088
2009.0    2686
2010.0    3412
2011.0    4379
2012.0    4565
2013.0    3494
2014.0    1162


In [18]:
before = len(df)

# Compute days_to_first_funding temporarily to filter
df['days_to_first_funding'] = (df['first_funding_at'] - df['founded_at']).dt.days

neg_mask = df['days_to_first_funding'] < 0
print(f"Rows with negative days_to_first_funding: {neg_mask.sum():,}")

df = df[df['days_to_first_funding'].isna() | (df['days_to_first_funding'] >= 0)]

log_step(12, 'Remove rows where funding predates founding', before, len(df),
         'days_to_first_funding < 0 means first_funding_at < founded_at — data entry error')

Rows with negative days_to_first_funding: 0
✅ [Step 12] Remove rows where funding predates founding
          44,493 → 44,493 rows  |  dropped 0  |  100.0% retained


In [19]:
before = len(df)

df = df.dropna(subset=['funding_total_usd'])

log_step(13, 'Drop rows missing funding_total_usd', before, len(df),
         'Core financial metric — cannot be accurately imputed for analysis')

⚠️ [Step 13] Drop rows missing funding_total_usd
          44,493 → 36,957 rows  |  dropped 7,536  |  83.1% retained


In [20]:
before = len(df)

df = df.dropna(subset=['founded_year'])

log_step(14, 'Drop rows missing founded_year', before, len(df),
         'Required for temporal/cohort analysis. Cannot be reliably imputed.')

⚠️ [Step 14] Drop rows missing founded_year
          36,957 → 28,534 rows  |  dropped 8,423  |  77.2% retained


In [21]:
before = len(df)

# ── 1. Speed to first funding (days) ──────────────────────────────────────────
# Already computed in Step 12; confirm it's still here
# (already in df from step 12)

# ── 2. Funding duration (first → last round, in days) ─────────────────────────
df['funding_duration_days'] = (df['last_funding_at'] - df['first_funding_at']).dt.days

# ── 3. Average funding per round ──────────────────────────────────────────────
df['avg_funding_per_round'] = np.where(
    df['funding_rounds'] > 0,
    df['funding_total_usd'] / df['funding_rounds'],
    0
)

# ── 4. Is US-based? ───────────────────────────────────────────────────────────
df['is_usa'] = (df['country_code'] == 'USA').astype(int)

# ── 5. Primary category (first pipe-separated category, skipping blank token) ─
df['primary_category'] = (
    df['category_list']
    .astype(str)
    .str.split('|')
    .apply(lambda parts: next((p.strip() for p in parts if p.strip() not in ['', 'nan', 'None']), np.nan))
)

# ── 6. Binary failure label ───────────────────────────────────────────────────
df['is_closed'] = (df['status'] == 'closed').astype(int)

# ── 7. Has reached Series A or beyond? ───────────────────────────────────────
series_cols = [c for c in ['round_A','round_B','round_C','round_D','round_E','round_F','round_G','round_H'] if c in df.columns]
df['reached_series_a'] = (df[series_cols].sum(axis=1) > 0).astype(int)

# ── 8. Founding decade ────────────────────────────────────────────────────────
df['founding_decade'] = (df['founded_year'] // 10 * 10).astype('Int64')

# ── 9. Funding tier (log-scale bucketed) ──────────────────────────────────────
# Useful for grouped failure-rate charts without raw dollar skew
def funding_tier(usd):
    if pd.isna(usd) or usd == 0: return 'Unknown'
    if usd < 100_000:            return '<$100K'
    if usd < 1_000_000:          return '$100K–$1M'
    if usd < 10_000_000:         return '$1M–$10M'
    if usd < 50_000_000:         return '$10M–$50M'
    return '$50M+'

df['funding_tier'] = df['funding_total_usd'].apply(funding_tier)

# ── 10. Has seed funding? ─────────────────────────────────────────────────────
if 'seed' in df.columns:
    df['has_seed'] = (df['seed'] > 0).astype(int)

log_step(15, 'Feature engineering', before, len(df),
         'Added: days_to_first_funding, funding_duration_days, avg_funding_per_round, '
         'is_usa, primary_category, is_closed, reached_series_a, founding_decade, '
         'funding_tier, has_seed')

print("\nNew features summary:")
new_feat_cols = ['days_to_first_funding','funding_duration_days','avg_funding_per_round',
                 'is_closed','reached_series_a','has_seed']
print(df[[c for c in new_feat_cols if c in df.columns]].describe().round(2).to_string())

✅ [Step 15] Feature engineering
          28,534 → 28,534 rows  |  dropped 0  |  100.0% retained

New features summary:
       days_to_first_funding  funding_duration_days  avg_funding_per_round  is_closed  reached_series_a  has_seed
count               28532.00               28532.00               28534.00   28534.00          28534.00  28534.00
mean                 1154.89                 408.63             4047894.36       0.05              0.34      0.36
std                  1403.47                 680.64             6580164.06       0.22              0.47      0.48
min                     0.00                   0.00                  14.00       0.00              0.00      0.00
25%                   242.00                   0.00              278945.25       0.00              0.00      0.00
50%                   619.00                   0.00             1322182.50       0.00              0.00      0.00
75%                  1525.00                 607.00             5071875.00       0

In [22]:
print("=" * 60)
print("FINAL DATASET QUALITY REPORT")
print("=" * 60)
print(f"\nShape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory:  {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\n── Status distribution (target variable) ──")
status_dist = df['status'].value_counts()
for status, count in status_dist.items():
    pct = count / len(df) * 100
    print(f"  {status:12s}: {count:>6,}  ({pct:.1f}%)")

print("\n── Failure rate (closed) by funding tier ──")
print(df.groupby('funding_tier')['is_closed'].mean().sort_values(ascending=False).round(3).to_string())

print("\n── Remaining missing values in key columns ──")
key_cols = ['status','funding_total_usd','country_code','founded_year',
            'funding_rounds','is_closed','primary_category']
missing_final = df[key_cols].isnull().sum()
print(missing_final.to_string())

print("\n── Top 10 markets (by startup count) ──")
if 'market' in df.columns:
    print(df['market'].value_counts().head(10).to_string())

FINAL DATASET QUALITY REPORT

Shape:   28,534 rows × 49 columns
Memory:  31.2 MB

── Status distribution (target variable) ──
  operating   : 24,694  (86.5%)
  acquired    :  2,390  (8.4%)
  closed      :  1,450  (5.1%)

── Failure rate (closed) by funding tier ──
funding_tier
<$100K       0.083
$100K–$1M    0.066
$1M–$10M     0.044
$10M–$50M    0.032

── Remaining missing values in key columns ──
status                  0
funding_total_usd       0
country_code         2029
founded_year            0
funding_rounds          0
is_closed               0
primary_category      923

── Top 10 markets (by startup count) ──
market
Software               3037
Biotechnology          2205
Mobile                 1265
E-Commerce             1013
Curated Web             960
Enterprise Software     876
Health Care             756
Hardware + Software     691
Advertising             674
Games                   673


In [23]:
etl_df = pd.DataFrame(etl_log)
print("=" * 80)
print("ETL PIPELINE AUDIT LOG")
print("=" * 80)
print(etl_df[['Step','Description','Before','After','Dropped','% Kept']].to_string(index=False))
print()
print(f"  Initial rows : {initial_rows:,}")
print(f"  Final rows   : {len(df):,}")
print(f"  Total dropped: {initial_rows - len(df):,}  ({(initial_rows - len(df)) / initial_rows * 100:.1f}% of raw data)")
print(f"  Rows retained: {len(df) / initial_rows * 100:.1f}%")

ETL PIPELINE AUDIT LOG
 Step                                                   Description  Before  After  Dropped  % Kept
    1                            Strip whitespace from column names   54294  54294        0   100.0
    2                          Convert funding_total_usd to numeric   54294  54294        0   100.0
    3                              Parse date columns to datetime64   54294  54294        0   100.0
    4                                 Enforce date logical ordering   54294  51555     2739    95.0
    5             Standardise string columns and unify null markers   51555  51555        0   100.0
    6                          Map status to 4 canonical categories   51555  51555        0   100.0
    7               Drop rows with missing status (target variable)   51555  45433     6122    88.1
    8 Remove duplicates on ['name', 'country_code', 'founded_year']   45433  45420       13   100.0
    9                Convert funding round columns to int (NaN → 0)   45420  

In [24]:
# ── Drop columns not needed for analysis ──────────────────────────────────────
cols_to_drop = ['permalink', 'homepage_url']   # administrative / URL fields
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ── Save to CSV ───────────────────────────────────────────────────────────────
df.to_csv(CLEAN_PATH, index=False)
print(f"✅ Cleaned dataset saved to: {CLEAN_PATH}")
print(f"   {len(df):,} rows × {df.shape[1]} columns")

# ── If running on Google Colab, also download it ──────────────────────────────
try:
    from google.colab import files
    files.download(CLEAN_PATH)
    print("📥 Download triggered (Colab).")
except ImportError:
    print("ℹ️  Not running on Colab — file saved locally.")

# ── Preview final columns ─────────────────────────────────────────────────────
print(f"\nFinal columns ({df.shape[1]}):")
print(df.columns.tolist())

✅ Cleaned dataset saved to: startups_cleaned.csv
   28,534 rows × 47 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download triggered (Colab).

Final columns (47):
['name', 'category_list', 'market', 'funding_total_usd', 'status', 'country_code', 'state_code', 'region', 'city', 'funding_rounds', 'founded_at', 'founded_month', 'founded_quarter', 'founded_year', 'first_funding_at', 'last_funding_at', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H', 'days_to_first_funding', 'funding_duration_days', 'avg_funding_per_round', 'is_usa', 'primary_category', 'is_closed', 'reached_series_a', 'founding_decade', 'funding_tier', 'has_seed']
